# Global CPG Market Data Cleaning

**Project question:** Which geographies and beverage categories are attractive for global CPG entry, based on comparable market values, volumes, packaging, and retail-channel trends from 2015 through 2025?

This notebook cleans four Euromonitor Passport exports in `data/`. The original files are read-only inputs; the script writes separate cleaned CSVs to `data/cleaned/`. Run this notebook from the repository or run `python datacleaning.py` from the project directory.

## Cleaning decisions

- Skip the five export preamble lines and remove only recognized source/date/copyright footer rows. Do not edit the original exports.
- Trim whitespace in text fields without changing meaningful category labels or geography names. The files contain no dedicated ID columns; row identity uses each table's geography/category/measure/packaging dimensions.
- Convert local-currency market values to **USD million** using annual World Bank official exchange rates (local currency units per US dollar). Keep existing USD values unchanged. Convert Japanese yen billions to yen millions before conversion.
- Standardize market volume labels to `metric tonnes` and `million liters`. These represent different physical dimensions and are not numerically combined. Pack sizes retain their physical dimension and use normalized `g` or `ml` labels; `Total` remains an aggregate, not a numeric size. Pack-volume units remain `million units`; channel shares remain percentages.
- Treat `-` in annual observation columns as unavailable, not as zero. Remove records with no observed year values because they cannot be responsibly imputed. For remaining numeric gaps, use row-wise cubic spline interpolation when at least four observed years bracket all missing years. Use linear interpolation with nearest-value extension at series boundaries if fewer than four values are available, the spline is invalid, or a spline estimate violates the data's non-negative / 0–100 bounds. Categorical/key fields are never imputed; missing keys stop the workflow with an explicit error.
- Record exact duplicate rows and repeated business keys. Only exact duplicate rows are removed; repeated keys are reported for review rather than silently discarded.

The pinned FX table is World Bank indicator [PA.NUS.FCRF](https://api.worldbank.org/v2/country/AUS/indicator/PA.NUS.FCRF?date=2015%3A2025&format=json), *Official exchange rate (LCU per US$, period average)*, annual 2015–2025 observations, retrieved 2026-10-02 (API metadata last updated 2026-07-13). Rates are embedded in `datacleaning.py` so the cleaning run does not depend on live network access. Cubic splines use SciPy `CubicSpline` and require SciPy from `requirements.txt`.

## 1. Capture source fingerprints and load the cleaner

In [6]:
from pathlib import Path
import hashlib
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "datacleaning.py").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not locate datacleaning.py from the notebook working directory.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import datacleaning as cleaning
import importlib
importlib.reload(cleaning)

SOURCE_FILES = list(cleaning.TABLES)
source_hashes_before = {
    name: hashlib.sha256((cleaning.DATA_DIR / name).read_bytes()).hexdigest()
    for name in SOURCE_FILES
}
print(f"Project directory: {PROJECT_ROOT}")
print(f"Source files: {SOURCE_FILES}")

Project directory: C:\Users\Ralph Jiang\dso576\cpg-global-entry-strategy
Source files: ['Market Sizes.csv', 'Pack Size.csv', 'Pack type.csv', 'Retail channels.csv']


## 2. Run the cleaning workflow

This cell recreates all four output CSVs and reports removed export footers, records with no observations, duplicate checks, missing-value treatment, currency conversions, and output locations.

In [7]:
cleaning_results = cleaning.clean_all()
summary = pd.DataFrame.from_dict(cleaning_results, orient="index")
display(summary)

,source_file,source_rows_before_footer_removal,footer_rows_removed,no_observation_rows_removed,cleaned_rows,exact_duplicate_rows_found,repeated_business_key_rows,missing_year_cells_before_imputation,missing_year_cells_after_imputation,missing_year_cells_imputed,cubic_spline_rows,linear_fallback_rows,imputation_method,currency_value_cells_converted,converted_geographies,cleaned_file
Market Sizes.csv,Market Sizes.csv,77,5,2,70,0,0,22,0,0,0,0,not needed after removing fully unavailable re...,187,"{'China': 2, 'India': 1, 'Japan': 2, 'Thailand...",data\cleaned\Market Sizes.csv
Pack Size.csv,Pack Size.csv,7213,4,89,7120,0,0,2758,0,1779,12,437,row-wise cubic spline; linear fallback when cu...,0,{},data\cleaned\Pack Size.csv
Pack type.csv,Pack type.csv,534,4,3,527,0,0,96,0,63,0,14,row-wise cubic spline; linear fallback when cu...,0,{},data\cleaned\Pack type.csv
Retail channels.csv,Retail channels.csv,869,5,286,578,0,0,3384,0,238,1,35,row-wise cubic spline; linear fallback when cu...,0,{},data\cleaned\Retail channels.csv


## 3. Verify outputs, units, and source preservation

In [8]:
cleaned = {
    name: pd.read_csv(cleaning.OUTPUT_DIR / name)
    for name in SOURCE_FILES
}

for name, frame in cleaned.items():
    assert len(frame) == cleaning_results[name]["cleaned_rows"], name
    assert not frame.isna().any().any(), f"Unexpected missing values in {name}"
    assert not frame.duplicated().any(), f"Exact duplicate rows remain in {name}"
    assert (frame[cleaning.YEARS].to_numpy(dtype=float) >= 0).all(), f"Negative values in {name}"
    assert cleaning_results[name]["missing_year_cells_after_imputation"] == 0, name
    assert cleaning_results[name]["cubic_spline_rows"] + cleaning_results[name]["linear_fallback_rows"] <= cleaning_results[name]["cleaned_rows"], name

market = cleaned["Market Sizes.csv"]
market_value_rows = market["Data Type"].str.contains("Value", case=False)
assert set(market.loc[market_value_rows, "Unit"]) == {"USD million"}
assert set(market.loc[market["Unit"].str.contains("tonnes|liters", case=False), "Unit"]) <= {
    "metric tonnes", "million liters"
}
pack_sizes = cleaned["Pack Size.csv"]["Pack Size"].astype(str)
normalized_sizes = pack_sizes.eq("Total") | pack_sizes.str.fullmatch(r"\d+(?:\.\d+)? (?:g|ml)")
assert normalized_sizes.all(), "Unexpected pack-size format"

channel_years = [year for year in cleaning.YEARS]
channel_values = cleaned["Retail channels.csv"][channel_years].to_numpy(dtype=float)
assert ((channel_values >= 0) & (channel_values <= 100)).all(), "Channel percentages outside 0-100"

source_hashes_after = {
    name: hashlib.sha256((cleaning.DATA_DIR / name).read_bytes()).hexdigest()
    for name in SOURCE_FILES
}
assert source_hashes_before == source_hashes_after, "A source CSV changed during cleaning"

print("PASS: all cleaned tables have complete year values, unique exact rows, and the expected row counts.")
print("PASS: market values use USD million; physical quantities are non-negative; pack sizes are normalized.")
print("PASS: interpolation leaves no missing year values and method counts are within row counts.")
print("PASS: channel percentages remain within 0-100.")
print("PASS: SHA-256 fingerprints confirm that all original source CSVs are unchanged.")

PASS: all cleaned tables have complete year values, unique exact rows, and the expected row counts.
PASS: market values use USD million; physical quantities are non-negative; pack sizes are normalized.
PASS: interpolation leaves no missing year values and method counts are within row counts.
PASS: channel percentages remain within 0-100.
PASS: SHA-256 fingerprints confirm that all original source CSVs are unchanged.


## 4. Check individual records

These checks compare source records with independently calculated expected outputs for local-currency conversion, existing USD values, volume labels, pack-size formatting, and retail-channel percentages.

In [9]:
from math import isclose

cubic_probe, cubic_method = cleaning._interpolate_year_series(
    pd.Series([0.0, 1.0, 8.0, 27.0, float("nan"), 125.0])
)
linear_probe, linear_method = cleaning._interpolate_year_series(
    pd.Series([0.0, float("nan"), 4.0, float("nan"), 8.0])
)
assert cubic_method == "cubic_spline" and isclose(cubic_probe.iloc[4], 64.0)
assert linear_method == "linear_fallback" and isclose(linear_probe.iloc[1], 2.0)
print("PASS: cubic-spline and insufficient-points linear-fallback examples.")

def first_matching(frame, conditions):
    mask = pd.Series(True, index=frame.index)
    for column, value in conditions.items():
        mask &= frame[column].astype(str).eq(str(value))
    matches = frame.loc[mask]
    if matches.empty:
        raise AssertionError(f"No source/output row matched {conditions}")
    return matches.iloc[0]


source_market = pd.read_csv(
    cleaning.DATA_DIR / "Market Sizes.csv",
    skiprows=cleaning.EXPORT_HEADER_ROWS,
    dtype="string",
    keep_default_na=False,
    encoding="utf-8-sig",
)

record_checks = []

for geography, year in [("Australia", "2025"), ("Japan", "2015")]:
    source_row = first_matching(
        source_market,
        {"Geography": geography, "Category": "Coffee", "Data Type": "Retail Value RSP"},
    )
    if source_row[year] == "-":
        raise AssertionError(f"Selected source record has no {year} observation: {geography}")
    output_row = first_matching(
        market,
        {"Geography": geography, "Category": "Coffee", "Data Type": "Retail Value RSP"},
    )
    currency = source_row["Unit"].split()[0]
    fx_series, expected_currency = cleaning.GEOGRAPHY_TO_CURRENCY[geography]
    assert currency == expected_currency
    magnitude = 1000 if source_row["Unit"].endswith("billion") else 1
    expected = float(source_row[year]) * magnitude / cleaning.FX_RATES_LCU_PER_USD[fx_series][int(year)]
    actual = float(output_row[year])
    record_checks.append((f"{geography} coffee retail value {year}", f"{source_row[year]} {source_row['Unit']}", expected, actual, output_row["Unit"]))

world_source = first_matching(
    source_market,
    {"Geography": "World", "Category": "Coffee", "Data Type": "Retail Value RSP"},
)
world_output = first_matching(
    market,
    {"Geography": "World", "Category": "Coffee", "Data Type": "Retail Value RSP"},
)
record_checks.append(("World coffee USD value unchanged (2025)", f"{world_source['2025']} {world_source['Unit']}", float(world_source["2025"]), float(world_output["2025"]), world_output["Unit"]))

source_pack = pd.read_csv(
    cleaning.DATA_DIR / "Pack Size.csv",
    skiprows=cleaning.EXPORT_HEADER_ROWS,
    dtype="string",
    keep_default_na=False,
    encoding="utf-8-sig",
)
source_pack_row = first_matching(
    source_pack,
    {"Geography": "USA", "Category": "Coffee", "Pack Size": "10 g"},
)
pack_output = first_matching(
    cleaned["Pack Size.csv"],
    {"Geography": "USA", "Category": "Coffee", "Pack Size": "10 g"},
)
record_checks.append(("USA coffee 10 g pack volume retained (2025)", f"{source_pack_row['2025']} {source_pack_row['Unit']}; {source_pack_row['Pack Size']}", float(source_pack_row["2025"]), float(pack_output["2025"]), f"{pack_output['Unit']}; {pack_output['Pack Size']}"))

source_channels = pd.read_csv(
    cleaning.DATA_DIR / "Retail channels.csv",
    skiprows=cleaning.EXPORT_HEADER_ROWS,
    dtype="string",
    keep_default_na=False,
    encoding="utf-8-sig",
)
source_channel_row = first_matching(
    source_channels,
    {"Geography": "World", "Category": "Coffee", "Outlet Type": "Retail Channels", "Data Type": "Retail Volume"},
)
channel_output = first_matching(
    cleaned["Retail channels.csv"],
    {"Geography": "World", "Category": "Coffee", "Outlet Type": "Retail Channels", "Data Type": "Retail Volume"},
)
record_checks.append(("World coffee retail-channel share retained (2025)", f"{source_channel_row['2025']}%", float(source_channel_row["2025"]), float(channel_output["2025"]), "%"))

record_check_table = pd.DataFrame(
    [
        {
            "Record check": label,
            "Original": original,
            "Expected": expected,
            "Actual": actual,
            "Unit": unit,
            "Matches": abs(expected - actual) <= max(1e-6, abs(expected) * 1e-10),
        }
        for label, original, expected, actual, unit in record_checks
    ]
)
display(record_check_table)
assert len(record_check_table) == 5
assert record_check_table["Matches"].all(), "An individual record did not match its expected result"

PASS: cubic-spline and insufficient-points linear-fallback examples.


,Record check,Original,Expected,Actual,Unit,Matches
0,Australia coffee retail value 2025,1389.8 AUD million,895.490908,895.490908,USD million,True
1,Japan coffee retail value 2015,365.1 JPY billion,3016.257911,3016.257911,USD million,True
2,World coffee USD value unchanged (2025),125307.9 USD million,125307.900000,125307.900000,USD million,True
3,USA coffee 10 g pack volume retained (2025),40.8 million units; 10 g,40.800000,40.800000,million units; 10 g,True
4,World coffee retail-channel share retained (2025),100%,100.000000,100.000000,%,True


## Cleaning decision log and decision verification

| Rule | Reason | Check |
|---|---|---|
| Skip five export preamble lines and remove recognized source/date/copyright footers. | Separate export metadata from business records without deleting valid observations. | The run removes 18 identified footer rows; source fingerprints confirm source files stay unchanged. |
| Trim surrounding whitespace, preserve identifiers and labels, and reject missing business keys. | Normalize formatting without guessing or changing record identity. | Each table's key columns are checked; missing keys raise an explicit error. |
| Treat `-` as unavailable, not zero; remove rows with no observed year values. | Zero is a real value and must not be substituted for unavailable observations. | 380 no-observation rows are removed; all remaining annual gaps are accounted for. |
| Use cubic spline only with at least four observations spanning the series; otherwise use linear fallback, rejecting invalid or out-of-range spline estimates. | Use interpolation only when supported and respect value bounds. | Probe checks yield 64 for a supported spline and 2 for a linear fallback; assertions check missing values, non-negativity, and 0–100 channel shares. |
| Convert local-currency market values to USD million using pinned annual World Bank rates; retain USD values. | Make market values comparable with transparent unit handling. | Independent Australia/AUD and Japan/JPY calculations are checked against cleaned values in the five-record table. |
| Standardize only comparable unit labels and normalize pack sizes to g/ml. | Preserve physical dimensions; tonnes, liters, units, and percentages are not interchangeable. | Notebook assertions validate unit labels and pack-size formats. |
| Remove exact duplicates; report repeated business keys rather than silently dropping them. | Avoid double-counting exact copies while preserving potentially distinct records. | Run found 0 exact duplicates and 0 repeated business-key rows. |
| Do not impute identifiers; fail for invalid numeric values or unsupported currency/unit mappings. | Avoid inventing values and expose unrecognized source data. | Cleaner raises explicit errors; output assertions check completed annual values and valid units. |

**Decision verification:** No Codex suggestion was used to choose these cleaning rules. I independently checked the currency-conversion decision: the source row reports Australia coffee retail value of AUD 1,389.8 million in 2025. Dividing by the pinned 2025 rate of 1.55199789 AUD per USD gives USD 895.490908 million, which matches the cleaned row. A second spot check converts Japan's JPY 365.1 billion in 2015 using `365.1 × 1,000 ÷ 121.04402568 = 3,016.257911` USD million; the cleaned output matches.

The data sample consists of selected source records in `data/sample/`, reproducible with `create_data_sample.py`. These are excerpts from licensed Passport exports; follow course/institution sharing terms.


## 5. Validate results and document limitations

**1. State the project vision and question: Done**

**Evidence:** The project compares geography and beverage-category opportunities using 2015–2025 market values, volumes, pack sizes, and retail-channel trends.

**Reference:** This notebook, *Project question* above.

**2. Define one row: Done**

**Evidence:** Each row represents a geography/category/measure record, with packaging or outlet dimensions where applicable; annual values are repeated observations for 2015–2025. The identifying columns are defined separately for each source table in `TABLES`.

**Reference:** `datacleaning.py`, `TABLES`.

**3. Preserve the source: Done**

**Evidence:** The four source CSVs are only read. SHA-256 fingerprints before and after cleaning match. Cleaned copies are written under `data/cleaned/`.

**Reference:** This notebook, *Capture source fingerprints* and *Verify outputs* cells; `data/cleaned/`.

**4. Inspect columns and types: Done**

**Evidence:** The four exports contain 16, 18, 17, and 15 columns, respectively. Their 2015–2025 fields are parsed as numeric; an unexpected non-numeric value stops the run with its location.

**Reference:** `datacleaning.py`, `clean_table`; the four source CSV headers in `data/`.

**5. Standardize text and units: Done**

**Evidence:** Text whitespace is trimmed and pack sizes are normalized to `g` or `ml`. Local-currency value observations are converted to USD million using annual-average exchange rates. Market volumes are labeled `metric tonnes` or `million liters`; channel values remain percentages, and pack volumes remain `million units`.

**Reference:** `datacleaning.py`, `_parse_pack_size` and `_convert_market_values`; *Verify outputs* and *Check individual records* cells above.

**6. Investigate duplicates: Done**

**Evidence:** Across all four tables, the run found 0 exact duplicate rows and 0 rows with repeated business keys. Exact duplicates would be removed; repeated keys are reported, not silently dropped.

**Reference:** The cleaning run summary above; `datacleaning.py`, `clean_table`.

**7. Handle missing data: Done**

**Evidence:** The run found 6,260 unavailable annual cells, including 380 records with no observed year values; those fully unavailable records were removed. The remaining 2,080 gaps were imputed: 13 rows used cubic spline and 486 rows used linear fallback. No annual values are missing in the cleaned outputs. Key fields are never imputed.

**Reference:** The cleaning run summary above; `datacleaning.py`, `_interpolate_year_series` and `clean_table`; `requirements.txt`.

**8. Check suspicious values: Done**

**Evidence:** Currency/geography mappings are validated, annual values are checked as numeric and non-negative, and retail-channel percentages are checked to remain between 0 and 100. The notebook assertions pass.

**Reference:** This notebook, *Verify outputs* cell; `datacleaning.py`, `_convert_market_values`.

**9. Verify merges: Not applicable**

**Evidence:** No tables are merged in this cleaning workflow.

**Reference:** `datacleaning.py`, `clean_all`.

**10. Reconcile changes: Done**

**Evidence:** The four exports contain 8,693 rows including footer text. Removing 18 footer rows leaves 8,675 business rows; removing 380 records with no year observations yields 8,295 cleaned rows. There are 0 exact duplicate rows. Per-file final counts are 70 Market Sizes, 7,120 Pack Size, 527 Pack type, and 578 Retail channels.

**Reference:** The cleaning run summary above, columns `footer_rows_removed`, `no_observation_rows_removed`, `exact_duplicate_rows_found`, and `cleaned_rows`.

**11. Check calculations: Done**

**Evidence:** The outputs contain no missing annual cells. Of the 2,080 imputed cells, 1,779 are in Pack Size, 63 in Pack type, and 238 in Retail channels. Five independent record checks match, including two converted local-currency values. Market value units are all `USD million`, and channel shares stay within 0–100.

**Reference:** This notebook, *Verify outputs* and *Check individual records* cells; the cleaning run summary above.

**12. Verify individual records: Done**

**Evidence:** Five checks pass: Australian and Japanese market values after USD conversion, an unchanged World USD value, a retained 10 g pack record, and a retained World retail-channel percentage. Each row includes the original, expected, and actual value.

**Reference:** This notebook, *Check individual records* cell and its record-check table.

**13. Test reproducibility: Done**

**Evidence:** The saved notebook code cells execute successfully. Rerunning the script produces byte-identical cleaned CSVs; pinned exchange rates make the run independent of live FX API availability.

**Reference:** `datacleaning.ipynb`; `datacleaning.py`; `requirements.txt`.

**14. Document remaining limitations: Done**

**Evidence:** Annual-average FX conversion does not adjust for inflation. Cubic-spline values are estimates and use linear fallback where spline interpolation is unsupported or yields invalid/out-of-range values. Records with no annual observation are excluded. Tonnes and liters remain distinct and should not be directly combined.

**Reference:** `datacleaning.py`, `_interpolate_year_series` and `clean_table`; World Bank FX source documented above.

The source exports and this cleaning process do not establish whether the original market estimates are representative. Interpret interpolated values and cross-market currency comparisons in light of these limitations.